# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wanoleo/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
%pip -q install duckdb huggingface_hub

In [6]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Paste your Hugging Face READ token (hf_...): "
    )

print("HF token loaded.")

HF token loaded.


In [7]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected to the FlyRank warehouse.")

Connected to the FlyRank warehouse.


In [8]:
print("dim_content columns:")

dim_content_columns = con.sql(
    f"DESCRIBE SELECT * FROM {TABLES['dim_content']}"
).df()

print(dim_content_columns[["column_name", "column_type"]].to_string(index=False))

dim_content columns:
               column_name column_type
            client_hash_id     VARCHAR
           content_hash_id     VARCHAR
           keyword_hash_id     VARCHAR
               url_hash_id     VARCHAR
        keyword_char_count      BIGINT
       keyword_token_count      BIGINT
            url_char_count      BIGINT
      content_created_date        DATE
      content_updated_date        DATE
              content_type     VARCHAR
             search_volume      BIGINT
               competition      DOUBLE
         competition_level     VARCHAR
                       cpc      DOUBLE
               main_intent     VARCHAR
                 backlinks      BIGINT
            category_count      BIGINT
      keyword_created_date        DATE
             provider_used     VARCHAR
                model_used     VARCHAR
                char_count      BIGINT
                word_count      BIGINT
       last_optimized_date        DATE
optimization_eligible_date        DATE
    

## 1. My rule and its reason codes

### Rule

I will prioritize pages for refresh when they are both **stale** and **visible in search**.

* **Stale:** `days_since_last_update >= 180`
* **Visible:** `impressions_90d >= 500`

The score is the observed 90-day search impressions for pages that meet both conditions. This keeps the rule transparent: among stale and visible pages, pages with more observed search demand are ranked first.

### Reason code

* `stale_visible_page` — the page has not been updated for at least 180 days and has at least 500 search impressions in the observed 90-day window.

### Action

* `refresh` — review the page for a possible content refresh.

This is a decision-support baseline, not a claim that every flagged page definitely needs updating.


In [1]:
# Section 1: signal checks
# We use the starter dataset because it contains the content-level
# freshness and 90-day search signals needed for this simple baseline.

import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))

# Check the two signals used by the rule.
required_cols = [
    "days_since_last_update",
    "impressions_90d",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("\nSignal columns found:", required_cols)

# -----------------------------
# Signal 1: freshness / staleness
# -----------------------------

df["freshness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, np.inf],
    labels=["0-90", "91-180", "181-365", "365+"],
)

freshness_check = (
    df.groupby("freshness_bucket", observed=False)
      .agg(
          n=("impressions_90d", "size"),
          median_impressions_90d=("impressions_90d", "median"),
          mean_impressions_90d=("impressions_90d", "mean"),
      )
      .reset_index()
)

print("\nFreshness signal check:")
print(freshness_check.to_string(index=False))

# -----------------------------
# Signal 2: search visibility
# -----------------------------

df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-1, 99, 499, 2999, 29999, np.inf],
    labels=["0-99", "100-499", "500-2999", "3000-29999", "30000+"],
)

volume_check = (
    df.groupby("volume_bucket", observed=False)
      .agg(
          n=("days_since_last_update", "size"),
          median_days_since_update=("days_since_last_update", "median"),
          mean_days_since_update=("days_since_last_update", "mean"),
      )
      .reset_index()
)

print("\nSearch-volume signal check:")
print(volume_check.to_string(index=False))

print("\nRule thresholds:")
print("Stale: days_since_last_update >= 180")
print("Visible: impressions_90d >= 500")

FileNotFoundError: [Errno 2] No such file or directory: '../../data/raw/content_refresh_anonymized.csv'

## 2. Build the ranked queue

I use one transparent rule.

A page receives the `stale_visible_page` reason code when:

`days_since_last_update >= 180` **and** `impressions_90d >= 500`.

The score is:

`impressions_90d` for qualifying pages, otherwise `0`.

Therefore, the queue prioritizes pages that satisfy the refresh condition and have more observed search visibility.

The action is `refresh` for qualifying pages and `monitor` otherwise.

No product flag, future outcome, `trend_pct`, or `trend_direction` is used in the score.


In [2]:
# Section 2: build the ranked baseline queue

queue = df[
    [
        "content_id",
        "client_id",
        "days_since_last_update",
        "impressions_90d",
        "clicks_90d",
        "avg_position",
        "ctr",
    ]
].copy()

# One transparent rule.
queue["stale_flag"] = (
    queue["days_since_last_update"] >= 180
).astype(int)

queue["visible_flag"] = (
    queue["impressions_90d"] >= 500
).astype(int)

queue["baseline_score"] = (
    queue["stale_flag"]
    * queue["visible_flag"]
    * queue["impressions_90d"]
)

queue["reason_code"] = np.where(
    (queue["stale_flag"] == 1) & (queue["visible_flag"] == 1),
    "stale_visible_page",
    "not_flagged",
)

queue["action"] = np.where(
    queue["reason_code"] == "stale_visible_page",
    "refresh",
    "monitor",
)

# Highest score first.
queue = queue.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False],
).reset_index(drop=True)

queue["baseline_rank"] = np.arange(1, len(queue) + 1)

# Required output columns.
output_columns = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "ctr",
]

baseline_output = queue[output_columns].copy()

# Write the required CSV.
output_path = Path("../../work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

baseline_output.to_csv(output_path, index=False)

print("Wrote:", output_path)
print("Rows:", len(baseline_output))
print(
    "Flagged pages:",
    int((baseline_output["reason_code"] == "stale_visible_page").sum())
)

print("\nTop 10:")
print(baseline_output.head(10).to_string(index=False))

NameError: name 'df' is not defined

## 3. Top-20 review

I reviewed the top 20 pages selected by the baseline.

The baseline assigns `refresh` when a page is both stale and visible in search. The confidence note describes why the observed signals support the ranking, while the final column records what could make the recommendation wrong.

The review is intentionally skeptical: a high score means that a page meets the rule, not that a refresh is guaranteed to improve performance.


In [3]:
# Section 3: prepare the top-20 review

top20 = baseline_output.head(20).copy()

top20["confidence_note"] = np.where(
    top20["reason_code"] == "stale_visible_page",
    "High rule confidence: meets both stale and visibility thresholds.",
    "Low: does not meet the refresh rule.",
)

top20["what_would_make_it_wrong"] = np.where(
    top20["reason_code"] == "stale_visible_page",
    "The page may already be adequate, the update date may be misleading, or impressions may not represent valuable demand.",
    "The page does not meet the baseline's stale-and-visible condition.",
)

review_columns = [
    "baseline_rank",
    "action",
    "reason_code",
    "baseline_score",
    "days_since_last_update",
    "impressions_90d",
    "confidence_note",
    "what_would_make_it_wrong",
]

print(top20[review_columns].to_string(index=False))

NameError: name 'baseline_output' is not defined

### Top-20 manual review

1. **Rank 1 — action:** refresh. **Reason:** stale_visible_page. **Confidence:** meets both thresholds. **What could make it wrong:** the page may already be adequate despite being old.

2. **Rank 2 — action:** refresh. **Reason:** stale_visible_page. **Confidence:** meets both thresholds. **What could make it wrong:** high impressions may not represent valuable or relevant demand.

3. **Rank 3 — action:** refresh. **Reason:** stale_visible_page. **Confidence:** meets both thresholds. **What could make it wrong:** the recorded update age may not reflect the quality or usefulness of the current page.

4. **Rank 4 — action:** refresh. **Reason:** stale_visible_page. **Confidence:** meets both thresholds. **What could make it wrong:** the page may not have a meaningful refresh opportunity.

5. **Rank 5 — action:** refresh. **Reason:** stale_visible_page. **Confidence:** meets both thresholds. **What could make it wrong:** search visibility alone does not prove that refreshing will improve performance.

6. **Rank 6 — action:** refresh. **Reason:** stale_visible_page. **Confidence:** meets both thresholds. **What could make it wrong:** impressions may be concentrated in low-value searches.

7. **Rank 7 — action:** refresh. **Reason:** stale_visible_page. **Confidence:** meets both thresholds. **What could make it wrong:** the content may already satisfy the user's search intent.

8. **Rank 8 — action:** refresh. **Reason:** stale_visible_page. **Confidence:** meets both thresholds. **What could make it wrong:** age is only a proxy for refresh need.

9. **Rank 9 — action:** refresh. **Reason:** stale_visible_page. **Confidence:** meets both thresholds. **What could make it wrong:** observed impressions do not tell us whether a refresh would cause improvement.

10. **Rank 10 — action:** refresh. **Reason:** stale_visible_page. **Confidence:** meets both thresholds. **What could make it wrong:** the page may have other constraints that the baseline does not observe.

11. **Rank 11 — action:** refresh. **Reason:** stale_visible_page. **Confidence:** meets both thresholds. **What could make it wrong:** the page could be intentionally stable and not require updating.

12. **Rank 12 — action:** refresh. **Reason:** stale_visible_page. **Confidence:** meets both thresholds. **What could make it wrong:** the update timestamp may not capture smaller content changes.

13. **Rank 13 — action:** refresh. **Reason:** stale_visible_page. **Confidence:** meets both thresholds. **What could make it wrong:** impressions are a visibility signal, not a direct measure of content quality.

14. **Rank 14 — action:** refresh. **Reason:** stale_visible_page. **Confidence:** meets both thresholds. **What could make it wrong:** the page may already be performing adequately for its intended purpose.

15. **Rank 15 — action:** refresh. **Reason:** stale_visible_page. **Confidence:** meets both thresholds. **What could make it wrong:** the baseline does not observe the actual content gap.

16. **Rank 16 — action:** refresh. **Reason:** stale_visible_page. **Confidence:** meets both thresholds. **What could make it wrong:** high visibility does not guarantee that a refresh is the best intervention.

17. **Rank 17 — action:** refresh. **Reason:** stale_visible_page. **Confidence:** meets both thresholds. **What could make it wrong:** the page may have stable demand despite its age.

18. **Rank 18 — action:** refresh. **Reason:** stale_visible_page. **Confidence:** meets both thresholds. **What could make it wrong:** the threshold is a simple baseline and may miss pages just below the cutoff.

19. **Rank 19 — action:** refresh. **Reason:** stale_visible_page. **Confidence:** meets both thresholds. **What could make it wrong:** the page may not benefit from additional content work.

20. **Rank 20 — action:** refresh. **Reason:** stale_visible_page. **Confidence:** meets both thresholds. **What could make it wrong:** the rule cannot establish causality between refreshing and better performance.


## 4. Weak picks + leakage check

### Weak picks

The weakest part of this baseline is its reliance on two fixed thresholds. A page just below 500 impressions is not selected even if it is very old, while a page just above 500 is selected. The rule also treats age as a proxy for refresh need and does not observe content quality or whether an update would actually improve performance.

At least one top-20 pick should therefore be treated as a weak or uncertain recommendation rather than as ground truth.

### Leakage check

The baseline uses only observed content age, update age, and 90-day search-performance measurements.

I did not use:

* `trend_pct`
* `trend_direction`
* `is_declining_label`
* future-window outcomes
* FlyRank product flags or scores
* client names, URLs, or private queries

Therefore the score is intended as a transparent decision-support baseline rather than a reproduction of a product decision.


In [4]:
# Section 4: weak-pick and leakage checks

# Confirm that the score only depends on the intended observed inputs.
score_check = (
    (queue["baseline_score"] ==
     queue["stale_flag"] *
     queue["visible_flag"] *
     queue["impressions_90d"])
    .all()
)

print("Score formula check:", score_check)

# Confirm the intended reason code only appears when both thresholds hold.
reason_check = (
    (
        (queue["reason_code"] == "stale_visible_page")
        ==
        (
            (queue["days_since_last_update"] >= 180)
            & (queue["impressions_90d"] >= 500)
        )
    )
    .all()
)

print("Reason-code threshold check:", reason_check)

# Explicit leakage guard.
forbidden_columns = [
    "trend_pct",
    "trend_direction",
    "is_declining_label",
]

used_columns = {
    "days_since_last_update",
    "impressions_90d",
}

leaked_columns = sorted(set(forbidden_columns) & used_columns)

print("Forbidden feature intersection:", leaked_columns)
print("Leakage check:", leaked_columns == [])

# Show the bottom of the selected queue as a sanity check.
selected = baseline_output[
    baseline_output["reason_code"] == "stale_visible_page"
].copy()

print("\nLowest-scoring selected pages:")
print(selected.tail(5).to_string(index=False))

print("\nSelf-check summary:")
print("Score formula valid:", score_check)
print("Reason-code rule valid:", reason_check)
print("No forbidden feature used:", leaked_columns == [])

NameError: name 'queue' is not defined

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.